# Prophet 2026 Projection & Early Evaluation

Mirrors notebook 07 (TimesFM 2026 projection) using **Facebook Prophet** with UK public holidays. Forecasts Jan–Mar 2026 at force level, then evaluates against actuals.

**Training context:** 2012–2019 + 2022–2025 (pandemic years excluded)  
**Forecast:** Jan–Mar 2026  
**Baseline:** 2025 monthly mean per force

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
from prophet import Prophet
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display

## 1. Load data

In [ ]:
FORCES = {
    'Cheshire Constabulary':         'Cheshire',
    'Lincolnshire Police':            'Lincolnshire',
    'Merseyside Police':              'Merseyside',
    'Metropolitan Police Service':    'Metropolitan',
    'West Midlands Police':           'West Midlands',
}
CRIME_TYPES = [
    'Violence and sexual offences', 'Criminal damage and arson', 'Drugs',
    'Violent crime', 'Burglary', 'Other theft', 'Vehicle crime',
    'Public order', 'Other crime', 'Shoplifting', 'Robbery',
    'Bicycle theft', 'Theft from the person', 'Possession of weapons',
    'Public disorder and weapons', 'Anti-social behaviour',
]
TRAIN_YEARS = set(range(2012, 2020)) | set(range(2022, 2026))
EVAL_MONTHS = pd.date_range('2026-01-01', periods=3, freq='MS')
force_labels = list(FORCES.values())

raw = pd.read_parquet('../data/processed/crimes_clean_dedup_all_years.parquet')
raw = raw[raw['Falls within'].isin(FORCES) & raw['Crime type'].isin(CRIME_TYPES)].copy()
raw['force'] = raw['Falls within'].map(FORCES)
raw['month'] = pd.to_datetime(raw['Month'])
raw = raw[raw['month'].dt.year >= 2012]

force_monthly = (
    raw.groupby(['force', 'month'])
    .size()
    .reset_index(name='count')
)
print(f'Date range: {force_monthly["month"].min().date()} → {force_monthly["month"].max().date()}')

## 2. Prophet parameters

In [ ]:
def prophet_forecast(train_series, n_months):
    """Fit Prophet on a monthly series and return n_months point forecasts."""
    df = train_series.reset_index()
    df.columns = ['ds', 'y']
    df['ds'] = pd.to_datetime(df['ds'])

    m = Prophet(
        seasonality_mode='multiplicative',
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        changepoint_prior_scale=0.15,
        seasonality_prior_scale=10,
        uncertainty_samples=0,
    )
    m.add_country_holidays(country_name='GB')
    m.fit(df)

    future = m.make_future_dataframe(periods=n_months, freq='MS', include_history=False)
    forecast = m.predict(future)
    return np.clip(forecast['yhat'].values[:n_months], 0, None)

## 3. Forecast Jan–Mar 2026

In [ ]:
forecasts = {}

for force in force_labels:
    print(f'Fitting {force}...', end='  ')
    train = (
        force_monthly[
            (force_monthly['force'] == force) &
            (force_monthly['month'].dt.year.isin(TRAIN_YEARS))
        ]
        .set_index('month')['count']
        .sort_index()
    )
    preds = prophet_forecast(train, 3)
    forecasts[force] = preds
    print(f'Jan={preds[0]:.0f}  Feb={preds[1]:.0f}  Mar={preds[2]:.0f}')

forecast_df = pd.DataFrame(
    {force: forecasts[force] for force in force_labels},
    index=EVAL_MONTHS.strftime('%Y-%m')
).T

print('\nProphet forecasts (Jan–Mar 2026):')
display(forecast_df.round(0).astype(int))

## 4. Evaluation vs actuals

In [ ]:
actuals_26 = (
    force_monthly[force_monthly['month'].isin(EVAL_MONTHS)]
    .pivot(index='force', columns='month', values='count')
)
actuals_26.columns = actuals_26.columns.strftime('%Y-%m')
actuals_26 = actuals_26.reindex(force_labels)

baseline_26 = (
    force_monthly[force_monthly['month'].dt.year == 2025]
    .groupby('force')['count'].mean()
    .reindex(force_labels)
)

# Load TimesFM Q1 results for comparison
try:
    tfm_eval = pd.read_csv('../outputs/timesfm_2026_Q1_eval.csv').set_index('force')
    has_tfm = True
except FileNotFoundError:
    has_tfm = False
    print('TimesFM Q1 results not found — run notebook 07 first for comparison')

# metric helpers (defined once, reused across the notebook)
def mae(pred, actual):
    return float(np.mean(np.abs(pred - actual)))

def rmse(pred, actual):
    return float(np.sqrt(np.mean((pred - actual) ** 2)))

def mape(pred, actual):
    # guard against division by zero where actual demand is 0
    return float(np.mean(np.abs((pred - actual) / np.where(actual == 0, 1, actual))) * 100)

rows = []
for force in force_labels:
    actual_vals   = actuals_26.loc[force].values.astype(float)
    prophet_vals  = forecasts[force]
    baseline_vals = np.full(3, baseline_26[force])

    row = {
        'force':        force,
        'baseline_mae': mae(baseline_vals, actual_vals),
        'prophet_mae':  mae(prophet_vals,  actual_vals),
        'prophet_rmse': rmse(prophet_vals, actual_vals),
        'prophet_mape': mape(prophet_vals, actual_vals),
    }
    if has_tfm:
        row['tfm_mae']  = tfm_eval.loc[force, 'model_mae']
        row['tfm_mape'] = tfm_eval.loc[force, 'model_mape']
    rows.append(row)

eval_df = pd.DataFrame(rows)
eval_df['delta_mae']    = eval_df['baseline_mae'] - eval_df['prophet_mae']
eval_df['prophet_rmae'] = eval_df['prophet_mae']  / eval_df['baseline_mae']
eval_df['prophet_wins'] = eval_df['prophet_mae']  < eval_df['baseline_mae']

cols = ['force','baseline_mae','prophet_mae','delta_mae','prophet_rmae','prophet_mape']
if has_tfm:
    cols += ['tfm_mae','tfm_mape']
display(
    eval_df[cols].round(2).set_index('force').rename(columns={
        'baseline_mae':'Baseline MAE','prophet_mae':'Prophet MAE',
        'delta_mae':'Δ MAE','prophet_rmae':'RMAE','prophet_mape':'Prophet MAPE%',
        'tfm_mae':'TimesFM MAE','tfm_mape':'TimesFM MAPE%',
    })
)

print(f"\nProphet — MAE: {eval_df['prophet_mae'].mean():.1f}  MAPE: {eval_df['prophet_mape'].mean():.1f}%  Win rate: {eval_df['prophet_wins'].mean()*100:.0f}%")
if has_tfm:
    print(f"TimesFM  — MAE: {eval_df['tfm_mae'].mean():.1f}  MAPE: {eval_df['tfm_mape'].mean():.1f}%")

## 5. Forecast vs Actual chart

In [ ]:
HISTORY_START = '2023-01-01'

fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharey=False)
fig.suptitle('Prophet 2026 Forecast vs Actual (Jan–Mar) — Force Level', fontsize=13, fontweight='bold')

for ax, force in zip(axes, force_labels):
    history = force_monthly[
        (force_monthly['force'] == force) &
        (force_monthly['month'] >= HISTORY_START) &
        (force_monthly['month'].dt.year.isin(TRAIN_YEARS))
    ].sort_values('month')

    ax.plot(history['month'], history['count'], color='#94a3b8', linewidth=1.2, label='History')
    ax.plot(EVAL_MONTHS, forecasts[force], color='#4CAF50', linewidth=2,
            marker='s', markersize=6, linestyle='--', label='Prophet', zorder=5)
    ax.plot(EVAL_MONTHS, actuals_26.loc[force].values, color='#FF5722', linewidth=2,
            marker='o', markersize=6, label='Actual', zorder=5)
    ax.axhline(baseline_26[force], color='#9E9E9E', linestyle=':', linewidth=1.2, label='2025 mean')

    ax.set_title(force, fontsize=10, fontweight='bold')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.tick_params(axis='x', labelsize=7, rotation=30)
    ax.tick_params(axis='y', labelsize=8)

    mape_val = eval_df[eval_df['force'] == force]['prophet_mape'].values[0]
    color = 'green' if mape_val < 5 else 'orange' if mape_val < 15 else 'red'
    ax.text(0.97, 0.97, f'MAPE {mape_val:.1f}%', transform=ax.transAxes,
            ha='right', va='top', fontsize=8, color=color, fontweight='bold')

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.05))
plt.tight_layout(rect=[0, 0.05, 1, 1])
os.makedirs('../outputs', exist_ok=True)
plt.savefig('../outputs/prophet_2026_Q1_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Side-by-side: Prophet vs TimesFM vs Baseline

In [ ]:
if has_tfm:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Prophet vs TimesFM — Jan–Mar 2026 Evaluation', fontsize=13, fontweight='bold')

    # MAE comparison
    x = np.arange(len(force_labels))
    w = 0.25
    axes[0].bar(x - w, eval_df['baseline_mae'], w, label='Baseline', color='#9E9E9E')
    axes[0].bar(x,     eval_df['prophet_mae'],  w, label='Prophet',  color='#4CAF50')
    axes[0].bar(x + w, eval_df['tfm_mae'],      w, label='TimesFM',  color='#2196F3')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(force_labels, rotation=15, ha='right')
    axes[0].set_title('MAE by Force (lower = better)')
    axes[0].set_ylabel('MAE')
    axes[0].legend()

    # MAPE comparison
    axes[1].bar(x - w/2, eval_df['prophet_mape'], w, label='Prophet', color='#4CAF50')
    axes[1].bar(x + w/2, eval_df['tfm_mape'],     w, label='TimesFM', color='#2196F3')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(force_labels, rotation=15, ha='right')
    axes[1].set_title('MAPE % by Force (lower = better)')
    axes[1].set_ylabel('MAPE %')
    axes[1].legend()

    plt.tight_layout()
    plt.savefig('../outputs/prophet_vs_timesfm_2026_Q1.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Run notebook 07 first to generate TimesFM Q1 results for comparison.')

## 7. Prophet components — seasonality insight

In [ ]:
# Show Prophet components for one force to understand what it learned
EXAMPLE_FORCE = 'Metropolitan'

train = (
    force_monthly[
        (force_monthly['force'] == EXAMPLE_FORCE) &
        (force_monthly['month'].dt.year.isin(TRAIN_YEARS))
    ]
    .set_index('month')['count']
    .sort_index()
)
df = train.reset_index()
df.columns = ['ds', 'y']

m = Prophet(
    seasonality_mode='multiplicative',
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    changepoint_prior_scale=0.15,
    seasonality_prior_scale=10,
    uncertainty_samples=0,
)
m.add_country_holidays(country_name='GB')
m.fit(df)

future = m.make_future_dataframe(periods=3, freq='MS')
forecast = m.predict(future)
fig = m.plot_components(forecast)
fig.suptitle(f'Prophet Components — {EXAMPLE_FORCE}', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/prophet_components_metropolitan.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Save

In [ ]:
eval_df.to_csv('../outputs/prophet_2026_Q1_eval.csv', index=False)
forecast_df.to_csv('../outputs/prophet_2026_Q1_forecasts.csv')
print('Saved to outputs/')